In [ ]:
import tensorflow as tf
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
import kagglehub

print("E.C.H.O. Diagnostic Environment Ready.")

In [ ]:
# Download latest version of the drone sound dataset
try:
    path = kagglehub.dataset_download("amineipad/drone-sound-audio-detection")
    print("Path to dataset files:", path)
except:
    print("KaggleHub error: Proceeding with existing local paths.")

In [ ]:
drone_path = os.path.join(path, 'Binary_Drone_Audio', 'yes_drone')
no_drone_path = os.path.join(path, 'Binary_Drone_Audio', 'unknown') 
drone_files = [os.path.join(drone_path, f) for f in os.listdir(drone_path) if f.endswith('.wav')]
no_drone_files = [os.path.join(no_drone_path, f) for f in os.listdir(no_drone_path) if f.endswith('.wav')]
print(f"Drone: {len(drone_files)} | No Drone: {len(no_drone_files)}")

In [ ]:
def extract_features(file_path, n_mels=32, target_frames=32):
    y, sr = librosa.load(file_path, sr=16000)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    S_dB = librosa.power_to_db(S, ref=np.max)
    if S_dB.shape[1] > target_frames: S_dB = S_dB[:, :target_frames]
    else: S_dB = np.pad(S_dB, ((0, 0), (0, target_frames - S_dB.shape[1])), mode='constant')
    return S_dB

all_files = drone_files + no_drone_files
all_labels = ["drone"] * len(drone_files) + ["no_drone"] * len(no_drone_files)
X = [extract_features(f) for f in all_files]
X_padded = np.array(X)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(all_labels)
y_categorical = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(X_padded, y_categorical, test_size=0.2, stratify=y_categorical)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train)
X_train_reshaped = X_train[..., np.newaxis]
X_val_reshaped = X_val[..., np.newaxis]
X_test_reshaped = X_test[..., np.newaxis]

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(32, 32, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(), 
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train_reshaped, y_train, validation_data=(X_val_reshaped, y_val), epochs=15, batch_size=32)

### 📊 Performance Analytics
Detailed metrics for drone sound detection.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_test_reshaped)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print("Classification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=label_encoder.classes_))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Acoustic Model Detection Matrix')
plt.ylabel('Ground Truth')
plt.xlabel('Prediction')
plt.show()

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
def representative_dataset():
    for f in X_train_reshaped[:100]:
        yield [f.astype(np.float32).reshape(1, 32, 32, 1)]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open('echo_acoustic_model.h', 'w') as f:
    f.write('/* TinyML Model for Project E.C.H.O. */\n')
    f.write('const unsigned char echo_acoustic_model_tflite[] = {\n ')
    for i, byte in enumerate(tflite_model):
        f.write(f'{byte:#04x}')
        if i != len(tflite_model) - 1: f.write(', ')
        if (i + 1) % 12 == 0: f.write('\n ')
    f.write('\n};')
print(f"SUCCESS: Model exported at {len(tflite_model)/1024:.1f} KB")